In [ ]:
import UQ_toolbox as uq
from medMNIST.utils import train_load_datasets_resnet as tr
import torch
import numpy as np
import torchvision.transforms as transforms
import numpy as np
import os
from torchvision import transforms

size = 224  # Image size for the models
batch_size = 4000  # Batch size for the DataLoader
model_global_perfs = {}
flag = 'breastmnist'
calib_method = 'platt'
color = False
uq_method = 'GPS'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
aug_folder = f'/mnt/data/psteinmetz/computer_vision_code/code/UQ_Toolbox/medMNIST/gps_augment/{size}*{size}/{flag}_calibration_set'

print(f"Processing {flag} with color={color}")
if color is True:
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[.5, .5, .5], std=[.5, .5, .5])
    ])
    
    transform_tta = transforms.Compose([
        transforms.ToTensor()
    ])
else:
    # For grayscale images, repeat the single channel to make it compatible with ResNet
    # ResNet expects 3 channels, so we repeat the single channel image
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.repeat(3, 1, 1)),
        transforms.Normalize(mean=[.5], std=[.5])
    ])
    
    transform_tta = transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.repeat(3, 1, 1))
        ])
    
    
models = tr.load_models(flag, device=device)
[study_dataset_plain, calibration_dataset, test_dataset], [train_loader, calibration_loader, test_loader], info = tr.load_datasets(flag, color, size, transform, batch_size, cache_test=False)
train_loaders, val_loaders = tr.CV_train_val_loaders(None, study_dataset_plain, batch_size=batch_size)
task_type = info['task']  # Determine the task type (binary-class or multi-class)
num_classes = len(info['label'])  # Number of classes
[_, calibration_dataset_tta, test_dataset_tta], [_, calibration_loader_tta, test_loader_tta], _ = tr.load_datasets(flag, color, size, transform_tta, batch_size)

results = tr.evaluate_model(models, test_loader, data_flag=flag, device=None, output_dir=None, prefix="test", display_cm=True)
y_true, y_scores, digits, correct_idx, incorrect_idx, indiv_scores = uq.evaluate_models_on_loader(models, test_loader, device)
results_calib = tr.evaluate_model(models, calibration_loader, data_flag=flag, device=None, output_dir=None, prefix="calib", display_cm=True)
y_true_calib, y_scores_calib, digits_calib, correct_idx_calib, incorrect_idx_calib, indiv_scores_calib = uq.evaluate_models_on_loader(models, calibration_loader, device)


In [ ]:
def computeTTA(aug_type, models, test_dataset, device, num_classes=2, correct_predictions_calibration=None, incorrect_predictions_calibration=None, image_normalization=False, aug_folder=None, mean=[0.5], std=[0.5], max_iterations=10, gps_augment=None, batch_size=None, color=False, im_size=None):
    if num_classes == 2:
        output_activation='sigmoid'
    else:
        output_activation='softmax'
    if aug_type == 'randaugment':
        # Original RandAugment (may include geometric transforms)
        transformation_pipeline = transforms.Compose([
            transforms.ToTensor(),
            transforms.ToPILImage(),
            transforms.RandAugment(num_ops=2, magnitude=9, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.PILToTensor(),
            transforms.ConvertImageDtype(torch.float),
            *([transforms.Normalize(mean=mean, std=std)] if image_normalization else []),
            *([transforms.Lambda(lambda x: x.repeat(3, 1, 1))] if color is False else [])
        ])
        metric, global_preds = uq.TTA(transformation_pipeline, models, test_dataset, device, nb_augmentations=5, nb_channels=3, output_activation=output_activation, usingBetterRandAugment=False, mean=mean, std=std, batch_size=batch_size)
    
    elif aug_type == 'crops_flips':
        transformation_pipeline = transforms.Compose([
            transforms.ToPILImage(),
            transforms.RandomResizedCrop(size=20, scale=(0.8, 1.0)),  # Random crop with resizing
            transforms.RandomHorizontalFlip(p=0.5),                   # Random horizontal flip
            transforms.RandomRotation(degrees=180),
            transforms.PILToTensor(),
            transforms.ConvertImageDtype(torch.float)
        ])
        metric, global_preds = uq.TTA(transformation_pipeline, models, test_dataset, device, nb_augmentations=5, nb_channels=3, output_activation=output_activation, usingBetterRandAugment=False, mean=mean, std=std, batch_size=batch_size)
        
    elif aug_type == 'GPS':
        if gps_augment is None:
            best_aug = uq.perform_greedy_policy_search(aug_folder, correct_predictions_calibration, incorrect_predictions_calibration, num_workers=90, max_iterations=max_iterations, num_searches=30, top_k=3, plot=True)
            if isinstance(best_aug, list) and all(isinstance(policy, list) for policy in best_aug):
                transformation_pipeline = []
                for aug in best_aug:
                    n, m, transformations = uq.extract_gps_augmentations_info(aug)
                    transformation_pipeline.append(transformations)
        else:
            n = gps_augment[0]
            m = gps_augment[1]
            transformation_pipeline = gps_augment[2]
        metric, global_preds_GPS = uq.TTA(transformation_pipeline, models, test_dataset, device, usingBetterRandAugment=True, n=n, m=m, nb_channels=3, image_size=im_size, image_normalization=image_normalization, output_activation=output_activation, mean=mean, std=std, batch_size=batch_size)
    
    return metric

In [ ]:
if models is not None and test_dataset_tta is not None and device is not None:
    metric = computeTTA('GPS', models, test_dataset_tta, device, num_classes=num_classes, correct_predictions_calibration=correct_idx, incorrect_predictions_calibration=incorrect_idx, aug_folder=aug_folder, max_iterations=10, gps_augment=None, im_size=size, batch_size=batch_size)